# Módulo 4 — Aprendizado Supervisionado (AIOps Foundation)

**Contexto:** no lab do Módulo 3, construímos o pipeline ETL (S3 → Glue → Athena) e,
na etapa de **Enriquecimento**, calculamos manualmente uma coluna `severity_score`
combinando o `level` do log com o `response_time_ms`, usando uma fórmula fixa que
nós mesmos escrevemos (veja `glue-job/etl_job.py`).

**A pergunta deste notebook:** será que um modelo consegue *aprender sozinho*,
a partir dos dados rotulados, uma lógica parecida com a que nós escrevemos à mão?
Isso é exatamente a definição de **Aprendizado Supervisionado** do manual: usar
dados de entrada e saída rotulados para que o modelo otimize e meça sua precisão
ao longo do tempo.

Vamos tratar isso como um problema de **Classificação** (um dos tipos de modelo
supervisionado citados no Módulo 4): prever se um evento é `critical` (1) ou não (0).


## 1. Carregar os dados processados

Exporte o CSV rodando `athena_export_query.sql` no Athena e salve como `logs_enriched_export.csv` nesta mesma pasta.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)


In [ ]:
df = pd.read_csv("logs_enriched_export.csv")
print(f"Total de registros: {len(df)}")
df.head()


## 2. Criar o rótulo (label) supervisionado

Vamos transformar `severity_score` em uma variável binária `is_critical`
(1 = deveria virar alerta prioritário, 0 = não). Esse é o **dado rotulado**
que o modelo vai usar para aprender — o "gabarito".

In [ ]:
df["is_critical"] = (df["severity_score"] >= 80).astype(int)
df["is_critical"].value_counts(normalize=True).rename("proporcao")


## 3. Experimento A — usando todas as features disponíveis

Features: `service`, `level`, `response_time_ms` (as duas últimas foram
literalmente usadas para *calcular* `severity_score` no Glue Job — então
esperamos que o modelo aprenda a lógica quase perfeitamente. Isso é
proposital: primeiro mostramos o modelo "acertando fácil" porque o sinal
está forte nos dados.

In [ ]:
features_a = pd.get_dummies(df[["service", "level", "response_time_ms"]], columns=["service", "level"])
X_a = features_a
y = df["is_critical"]

X_train_a, X_test_a, y_train, y_test = train_test_split(
    X_a, y, test_size=0.25, random_state=42, stratify=y
)

model_a = DecisionTreeClassifier(max_depth=4, random_state=42)
model_a.fit(X_train_a, y_train)

pred_a = model_a.predict(X_test_a)
print("Acurácia (Experimento A):", round(accuracy_score(y_test, pred_a), 3))
print()
print(classification_report(y_test, pred_a))


In [ ]:
cm_a = confusion_matrix(y_test, pred_a)
print("Matriz de confusão (Experimento A):")
print(cm_a)


**Ponto de discussão em aula:** a acurácia sai em ~100%. Pergunte à turma:
*"isso significa que o modelo é ótimo, ou que demos 'cola' para ele?"* — a
resposta certa é a segunda: o rótulo `is_critical` foi construído a partir
dessas mesmas features (`level` + `response_time_ms`), então o modelo está
basicamente "redescobrindo" a nossa fórmula. Um resultado perfeito em ML
supervisionado é motivo de **desconfiança**, não de comemoração.

## 4. Experimento B — sem `level` e sem `response_time_ms` (só `service`)

Agora simulamos um cenário em que as variáveis que carregam o sinal **não estão
disponíveis** (ex: uma fonte de dados que falhou, ou um sistema legado que não
expõe nível de log nem tempo de resposta). Sobra só `service`.

> Por que removemos as duas? Porque neste dataset cada uma sozinha já
> basta para acertar quase tudo: erros têm `level = ERROR` *e* tempos de
> resposta altos (800-6000 ms contra no máximo ~250 ms nos demais). Tirar só
> uma delas não muda a acurácia — as features são **redundantes**.

In [ ]:
features_b = pd.get_dummies(df[["service"]], columns=["service"])
X_b = features_b

X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_b, y, test_size=0.25, random_state=42, stratify=y
)

model_b = DecisionTreeClassifier(max_depth=4, random_state=42)
model_b.fit(X_train_b, y_train_b)

pred_b = model_b.predict(X_test_b)
print("Acurácia (Experimento B, só service):", round(accuracy_score(y_test_b, pred_b), 3))

# Referência: um "modelo" que sempre responde "não crítico" (sem aprender nada)
baseline = 1 - y_test_b.mean()
print("Baseline (sempre prever 'não crítico'):", round(baseline, 3))
print()
print(classification_report(y_test_b, pred_b, zero_division=0))


**Ponto de discussão em aula:** a acurácia cai de ~100% para algo perto de
84% — praticamente igual ao baseline, e repare no `recall` da classe `1`: o
modelo quase não encontra os eventos críticos. Isso mostra duas coisas:

1. **A qualidade e a disponibilidade dos dados de entrada limitam o modelo.**
   Sem `level` e `response_time_ms`, não há sinal para aprender — conecte com o
   Módulo 3 e com o Módulo 8: *"não dedique tempo a ML até que dados sólidos
   estejam disponíveis"*.
2. **Acurácia sozinha engana em dados desbalanceados.** Só ~16% dos eventos são
   críticos, então "sempre dizer que não é crítico" já acerta ~84%. O que
   importa aqui é `recall`/`precision` da classe crítica.

**Pergunta extra:** por que tirar só `response_time_ms` (ou só `level`) *não*
derrubou a acurácia? (Resposta: features redundantes — cada uma carrega o mesmo
sinal. Em dados reais isso é comum, e é por isso que remover uma variável nem
sempre muda o resultado.)

## 5. Interpretabilidade — o que o modelo aprendeu?

Uma vantagem de Árvores de Decisão (vs. modelos mais complexos como redes neurais) é conseguirmos *ler* a lógica aprendida — importante em AIOps, onde o time de operações precisa confiar na decisão do modelo.

In [ ]:
print(export_text(model_a, feature_names=list(X_a.columns), max_depth=3))


In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))
plot_tree(model_a, feature_names=list(X_a.columns), class_names=["not_critical", "critical"],
          filled=True, max_depth=3, fontsize=8, ax=ax)
plt.title("Árvore de decisão aprendida (Experimento A, profundidade limitada para leitura)")
plt.show()


## 6. Perguntas para conduzir a discussão em sala

1. O que aconteceria se o dataset de treino tivesse muito mais eventos `INFO`
   do que `ERROR` (desbalanceamento de classes)? Isso é comum em ambientes
   reais — a maioria dos eventos não é crítica.
2. Esse modelo foi treinado com uma fórmula que *nós* escrevemos como rótulo.
   Numa empresa real, quem definiria o que é "crítico"? Como isso afeta o
   viés do modelo (conceito do Módulo 8 — Viés em Machine Learning)?
3. Esse é um problema de **Classificação**. Como ficaria se, em vez de prever
   `is_critical` (0/1), quiséssemos prever o `severity_score` contínuo
   (0-100)? (Resposta: seria **Regressão** — outro tipo de modelo
   supervisionado citado no manual. Fica como exercício trocar
   `DecisionTreeClassifier` por `DecisionTreeRegressor`.)
4. Isso foi aprendizado **supervisionado** porque tínhamos rótulos
   (`severity_score`/`is_critical`). Que tipo de pergunta, sobre esse mesmo
   dataset de logs, só poderia ser respondida com aprendizado **não
   supervisionado** (ex: Clustering)? (Gancho para revisar a diferença
   Supervisionado x Não Supervisionado do próprio Módulo 4.)
